# Global and Sequential Comparison

Compare $N_{\mathrm{greedy}}^{\mathrm{seq}}$, $N_{\min}^{\mathrm{seq}}$, and $N_{\min}^{\mathrm{global}}$ on the saved one- and two-dimensional datasets. This notebook runs independently from the inputs.


## Saved-result reuse

With `REUSE_SAVED_RESULTS = True`, matching DP results from 01 or earlier 02/03 runs are reused. Missing or stale cases are solved normally. Metadata, input freshness or hashes, and allocation feasibility are checked. Let 01 finish first for maximum reuse; neither notebook requires it.

Greedy and the global integer program are solved for every selected case.


## Budgets

The global integer-program minimum is

$$N_{\min}^{\mathrm{global}}=\min_{\boldsymbol n\in\mathbb Z_{\geq0}^M}\sum_m n_m,
\qquad \sum_m n_m|r_{ijm}|\geq\pi\quad\text{for every }i<j.$$

Here $r_{ijm}$ is the principal phase gap. Sequential DP gives $N_{\min}^{\mathrm{seq}}$; Greedy gives $N_{\mathrm{greedy}}^{\mathrm{seq}}$. Each case satisfies
$N_{\min}^{\mathrm{global}}\leq N_{\min}^{\mathrm{seq}}\leq N_{\mathrm{greedy}}^{\mathrm{seq}}$.

The global minimum refers to this integer program; its constraints alone do not certify a shared global input state. HiGHS uses zero relative MIP gap and $10^{-9}$ feasibility tolerances; allocations are independently checked to $10^{-8}$ after normalization by $\pi$.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

working_directory = Path.cwd().resolve()
PROJECT_DIRECTORY = (
    working_directory / "problem-1-zero-error"
    if (working_directory / "problem-1-zero-error").is_dir()
    else working_directory
)
SOURCE_DIRECTORY = PROJECT_DIRECTORY / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))
for module in ("run_config", "sensor_budget_analysis", "global_minimum", "phase_data", "optimal", "eval"):
    sys.modules.pop(module, None)
from sensor_budget_analysis import (
    run_budget_analysis, plot_budget_comparison, configure_style, save_figure,
)
print("Project directory:", PROJECT_DIRECTORY)


## Run configuration

Set `RUN_MODE` to `"test"` (132 cases: all families and K, M=3 or 6, replicate 1) or `"full"` (39,600 cases). Run all cells after changing it. Test and full outputs are separate; final figures use full results.


In [ ]:
RUN_MODE = "full"  # "test" or "full"
REUSE_SAVED_RESULTS = True  # False forces a fresh DP solve for every selected case.

if RUN_MODE not in ("test", "full"):
    raise ValueError('RUN_MODE must be "test" or "full"')
print("Run mode:", RUN_MODE)
print("Expected standard benchmark cases:", 132 if RUN_MODE == "test" else 39_600)


## Compute budgets


In [ ]:
analysis = run_budget_analysis(
    PROJECT_DIRECTORY, include_global_minimum=True, run_mode=RUN_MODE,
    reuse_saved=REUSE_SAVED_RESULTS,
)
rows = analysis["rows"]
summary = analysis["summary"]
OUTPUT_DIRECTORY = analysis["output_directory"]
assert all(row["N_min_global"] <= row["N_min"] <= row["N_greedy"] for row in rows)
print(f"Verified all {len(rows):,} paired instances.")
print("Output:", OUTPUT_DIRECTORY)

## Budget summary

All three curves average the same cases, pooling families and M at each K: 1,800 cases per dimension in full mode, 6 in test mode. The fixed-K,M table keeps location counts separate.


In [ ]:
def show_summary(rows, include_global=False):
    header = r"| Dimension | K | Cases | Mean $N_{\min}^{\mathrm{seq}}$ |"
    separator = "|---|---:|---:|---:|"
    if include_global:
        header += r" Mean $N_{\mathrm{greedy}}^{\mathrm{seq}}$ | Mean $N_{\min}^{\mathrm{global}}$ | Mean gap | Equal budgets (%) |"
        separator += "---:|---:|---:|---:|"
    lines = [header, separator]
    for row in rows:
        line = (f"| {row['dimension']} | {row['K']} | {row['case_count']} | "
                f"{row['mean_N_min_opt']:.3f} |")
        if include_global:
            line += (f" {row['mean_N_greedy']:.3f} | {row['mean_N_min_global']:.3f} | {row['mean_sensor_gap']:.3f} | "
                     f"{100 * row['equal_budget_fraction']:.1f} |")
        lines.append(line)
    display(Markdown("\n".join(lines)))

show_summary(summary, include_global=True)

## Budget comparison


In [ ]:
figure = plot_budget_comparison(summary)
if RUN_MODE == "test":
    figure.suptitle(f"Test run: {len(rows):,} cases (subset of benchmark)")
    figure.tight_layout(rect=(0, 0, 1, 0.94))
save_figure(figure, OUTPUT_DIRECTORY / "figures", "global_vs_seq_by_k")
display(figure)
plt.close(figure)

## Saved outputs

Results go to `results/global/{test,full}/`. `tables/case_budgets.csv` stores budgets, allocations, timings, and diagnostics. `global_vs_seq_by_dimension_k.csv` and `global_vs_seq_by_k_m.csv` contain summaries; `figures/global_vs_seq_by_k.{pdf,png}` contains the comparison. Final figures read the full summary.
